In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, classification_report

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

SEARCH_PATHS = [
    (Path.cwd().parent / "data" / "flows.csv"),
    Path("mirai_flows.csv"),
    Path("Mirai.csv"),
    Path.home() / "Downloads" / "mirai_flows.csv",
    Path.home() / "datasets" / "mirai_flows.csv",
]

CSV_PATH = next((p for p in SEARCH_PATHS if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError(
        "Dataset Mirai no encontrado.\n"
        "Columna objetivo esperada: 'label' con valores 'benign' / nombre_ataque."
    )

df = pd.read_csv(CSV_PATH)
print(f"Shape: {df.shape}")
print(df['label'].value_counts())

Shape: (1006776, 13)
label
1    998983
0      7793
Name: count, dtype: int64


In [2]:
df = pd.read_csv(CSV_PATH)
df['src_ip'] = df['src_ip'].astype(str)

# Estado reclasificado semánticamente (igual que Markov Mirai)
def reclassify_state(row):
    proto  = row['state']
    b_pkts = row['b_pkts']
    avg_ps = row['avg_pkt_size']
    if proto == 'DNS':               return 'DNS_FLOOD'
    if proto in ('HTTP', 'HTTPS'):   return 'HTTP_FLOOD'
    if proto == 'SSH':               return 'OTHER'
    if proto == 'UDP_OTHER':
        if b_pkts == 0 and avg_ps < 100: return 'UDP_SMALL_NORESPONSE'
        elif b_pkts == 0:                return 'UDP_LARGE_NORESPONSE'
        else:                            return 'UDP_BIDIRECTIONAL'
    if proto == 'TCP_OTHER':
        if b_pkts == 0 and avg_ps < 80: return 'TCP_SYN_LIKE'
        elif b_pkts == 0:               return 'TCP_ACK_LIKE'
        else:                           return 'TCP_ESTABLISHED'
    return 'OTHER'
df['state'] = df.apply(reclassify_state, axis=1)

# Subsampleo por clase — igual que Markov Mirai (Hwang Tabla 12)
HWANG_CLASS_TOTAL = {
    'Normal':       68200 + 8525,
    'ACK_Flood':     6600 +  825,
    'DNS_Flood':     4312 +  539,
    'Mirai_CnC':    68200 + 8525,
    'GREIP_Flood':  24712 + 3089,
    'HTTP_Flood':     120 +   15,
    'SYN_Flood':    68200 + 8525,
    'UDP_Flood':    28816 + 3062,
    'VSE_Flood':     4432 +  554,
}

np.random.seed(42)
idx_keep_list = []
for cls_name, n_total in HWANG_CLASS_TOTAL.items():
    idx_cls = np.where(df['class_name'].values == cls_name)[0]
    if len(idx_cls) == 0:
        print(f'  AVISO: clase {cls_name!r} no encontrada')
        continue
    if len(idx_cls) > n_total:
        idx_cls = np.random.choice(idx_cls, n_total, replace=False)
    idx_keep_list.append(idx_cls)
idx_keep = np.sort(np.concatenate(idx_keep_list))
df = df.iloc[idx_keep].reset_index(drop=True)

# Features (igual que Markov Mirai)
FEATURE_COLS = ['n_pkts', 'n_bytes', 'f_pkts', 'f_bytes',
                'b_pkts', 'b_bytes', 'avg_pkt_size', 'duration', 'state']
df['state'] = LabelEncoder().fit_transform(df['state'].astype(str))

X = df[FEATURE_COLS].values.astype(np.float32)
y = df['label'].values.astype(int)

# Split 64/16/20 (necesita validación para GridSearchCV)
X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED, stratify=y)
X_tr, X_val, y_tr, y_val   = train_test_split(X_tv, y_tv, test_size=0.20, random_state=SEED, stratify=y_tv)

# MinMaxScaler post-split (igual que Autoencoder Mirai)
scaler = MinMaxScaler()
X_tr_scaled  = scaler.fit_transform(X_tr)
X_val_scaled = scaler.transform(X_val)
X_te_scaled  = scaler.transform(X_test)

N_FEATURES = X_tr_scaled.shape[1]
print(f"Train: {X_tr_scaled.shape} | Val: {X_val_scaled.shape} | Test: {X_te_scaled.shape}")
print(f"Normal (0): {(y==0).sum():,}  |  Ataque (1): {(y==1).sum():,}")

Train: (103285, 9) | Val: (25822, 9) | Test: (32277, 9)
Normal (0): 7,793  |  Ataque (1): 153,591


In [3]:
def build_autoencoder(input_dim, latent_dim=3):
    inputs = keras.Input(shape=(input_dim,))
    x = layers.Dense(32, activation='tanh')(inputs)
    x = layers.Dense(24, activation='tanh')(x)
    x = layers.Dense(12, activation='tanh')(x)
    x = layers.Dense(6,  activation='tanh')(x)
    latent  = layers.Dense(latent_dim, activation='tanh', name='latent')(x)
    x = layers.Dense(6,  activation='tanh')(latent)
    x = layers.Dense(12, activation='tanh')(x)
    x = layers.Dense(24, activation='tanh')(x)
    x = layers.Dense(32, activation='tanh')(x)
    outputs = layers.Dense(input_dim, activation='linear')(x)
    autoencoder = Model(inputs, outputs, name='AE_Van2020')
    encoder     = Model(inputs, latent,  name='Encoder_Van2020')
    return autoencoder, encoder

autoencoder, encoder = build_autoencoder(input_dim=N_FEATURES, latent_dim=3)
autoencoder.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mse')

early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)
autoencoder.fit(
    X_tr_scaled, X_tr_scaled,
    validation_data=(X_val_scaled, X_val_scaled),
    epochs=50, batch_size=256,
    callbacks=[early_stop], verbose=1
)

Z_train = encoder.predict(X_tr_scaled,  verbose=0)
Z_val   = encoder.predict(X_val_scaled, verbose=0)
Z_test  = encoder.predict(X_te_scaled,  verbose=0)
print(f"Espacio latente — Train: {Z_train.shape} | Val: {Z_val.shape} | Test: {Z_test.shape}")

dt_gs = GridSearchCV(
    DecisionTreeClassifier(random_state=SEED),
    param_grid={'max_depth': [3, 5, 10, 15, None], 'min_samples_split': [2, 5, 10], 'criterion': ['gini', 'entropy']},
    cv=5, scoring='f1', n_jobs=-1
)
dt_gs.fit(Z_train, y_tr)
best_dt = dt_gs.best_estimator_
print(f"DT  — mejores params: {dt_gs.best_params_} | CV F1: {dt_gs.best_score_:.4f}")

knn_gs = GridSearchCV(
    KNeighborsClassifier(),
    param_grid={'n_neighbors': [3, 5, 7, 10, 15], 'metric': ['euclidean', 'manhattan'], 'weights': ['uniform', 'distance']},
    cv=5, scoring='f1', n_jobs=-1
)
knn_gs.fit(Z_train, y_tr)
best_knn = knn_gs.best_estimator_
print(f"kNN — mejores params: {knn_gs.best_params_} | CV F1: {knn_gs.best_score_:.4f}")

best_f1, best_svm, best_C = -1, None, None
for C in [0.01, 0.1, 1.0, 10.0]:
    clf = CalibratedClassifierCV(LinearSVC(C=C, max_iter=2000, random_state=SEED), cv=3)
    clf.fit(Z_train, y_tr)
    f1 = f1_score(y_val, clf.predict(Z_val), pos_label=1, average='binary')
    if f1 > best_f1:
        best_f1, best_svm, best_C = f1, clf, C
print(f"SVM — mejor C: {best_C} | F1 val: {best_f1*100:.2f}%")

classifiers = {'DT': best_dt, 'kNN': best_knn, 'SVM': best_svm}

Epoch 1/50
404/404 [==============================] - 5s 7ms/step - loss: 0.0018 - val_loss: 8.5417e-05
Epoch 2/50
404/404 [==============================] - 1s 3ms/step - loss: 5.2437e-05 - val_loss: 3.2413e-05
Epoch 3/50
404/404 [==============================] - 1s 3ms/step - loss: 2.9159e-05 - val_loss: 2.4969e-05
Epoch 4/50
404/404 [==============================] - 1s 3ms/step - loss: 2.3061e-05 - val_loss: 1.9457e-05
Epoch 5/50
404/404 [==============================] - 1s 3ms/step - loss: 2.1081e-05 - val_loss: 1.9838e-05
Epoch 6/50
404/404 [==============================] - 1s 3ms/step - loss: 2.0142e-05 - val_loss: 2.3426e-05
Epoch 7/50
404/404 [==============================] - 1s 3ms/step - loss: 2.0104e-05 - val_loss: 2.1796e-05
Epoch 8/50
404/404 [==============================] - 1s 3ms/step - loss: 1.8793e-05 - val_loss: 1.7030e-05
Epoch 9/50
404/404 [==============================] - 1s 3ms/step - loss: 1.8906e-05 - val_loss: 2.4758e-05
Epoch 10/50
404/404 [===========

In [4]:
SEP = '─' * 58
print("Autoencoder + DT/kNN/SVM — Dataset Mirai")
print(SEP)
print(f"{'Modelo':<6} {'Prec':>6} {'Rec':>6} {'F1':>6} {'AUC':>6}")
print(f"{'─'*6} {'─'*6} {'─'*6} {'─'*6} {'─'*6}")

for name, clf in classifiers.items():
    y_pred = clf.predict(Z_test)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score   (y_test, y_pred, zero_division=0)
    f1   = f1_score       (y_test, y_pred, pos_label=1, average='binary', zero_division=0)
    try:
        auc = roc_auc_score(y_test, clf.predict_proba(Z_test)[:, 1])
    except Exception:
        auc = float('nan')
    print(f"{name:<6} {prec:.3f}  {rec:.3f}  {f1:.3f}  {auc:.3f}")

print(SEP)
print(classification_report(y_test, best_dt.predict(Z_test), target_names=['Benign', 'Malicious'], zero_division=0))

Autoencoder + DT/kNN/SVM — Dataset Mirai
──────────────────────────────────────────────────────────
Modelo   Prec    Rec     F1    AUC
────── ────── ────── ────── ──────
DT     0.999  0.999  0.999  0.995
kNN    0.999  0.999  0.999  0.998
SVM    0.968  1.000  0.984  0.862
──────────────────────────────────────────────────────────
              precision    recall  f1-score   support

      Benign       0.98      0.99      0.98      1559
   Malicious       1.00      1.00      1.00     30718

    accuracy                           1.00     32277
   macro avg       0.99      0.99      0.99     32277
weighted avg       1.00      1.00      1.00     32277

